In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
# allow importing the "backbone" package when the kernel's cwd is notebooks/
sys.path.append(os.path.abspath(".."))
import backbone.custom_metrics.AstroMLmod3 as AstroMLmod
import backbone.custom_metrics.Intrinsic_dimension as id_module
from backbone.data_handle.Test import KNN_accuracy
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import davies_bouldin_score

# BYOL & DINO test representations: TPCF & ID scores per epoch

In [ ]:
import importlib
from sklearn.decomposition import IncrementalPCA
importlib.reload(AstroMLmod)

data_dir = "/idia/projects/camil/Koketso/data_"
models = ["byol", "dino"]
results = {}

for model in models:
    prefix = f"{model}_galaxy10_val_representations_epoch_"
    epochs = sorted(int(f[len(prefix):-4]) for f in os.listdir(data_dir) if f.startswith(prefix))
    rows = []

    for epoch in epochs:
        df = pd.read_csv(f"{data_dir}/{prefix}{epoch}.csv")
        representations = df.iloc[:, 2:].to_numpy()
        labels = df["label"].to_numpy()
        n = representations.shape[0]

        # TPCF_score bootstraps over subsamples and returns (mean, std)
        tpcf, tpcf_std = AstroMLmod.TPCF_score(representations, epoch=epoch)
        id_ = id_module.id_score(representations)
        db_score = davies_bouldin_score(representations, labels)
        knn_acc, knn_acc_std = KNN_accuracy(representations, labels)

        # average pairwise cosine similarity, excluding self-similarity on the diagonal
        cos_sim_matrix = cosine_similarity(representations)
        off_diag_mask = ~np.eye(n, dtype=bool)
        avg_cos_sim = cos_sim_matrix[off_diag_mask].mean()
        std_cos_sim = cos_sim_matrix[off_diag_mask].std()

        # anisotropy: fraction of total variance explained by the top principal component,
        # on unit-normalised vectors so a single extreme-norm point cannot dominate the variance
        unit_representations = representations / np.linalg.norm(representations, axis=1, keepdims=True)
        anisotropy = IncrementalPCA(n_components=1).fit(unit_representations).explained_variance_ratio_[0]

        rows.append({
            "epoch": epoch,
            "tpcf_score": tpcf,
            "tpcf_std": tpcf_std,
            "id_mean": id_[0],
            "id_std": id_[1],
            "avg_cosine_similarity": avg_cos_sim,
            "cosine_similarity_std": std_cos_sim,
            "anisotropy": anisotropy,
            "davies_bouldin": db_score,
            "knn_accuracy": knn_acc,
            "knn_accuracy_std": knn_acc_std,
        })

        print(f"{model} epoch {epoch}: TPCF={tpcf:.4f}±{tpcf_std:.4f} ID={id_} CosineSim={avg_cos_sim:.4f}±{std_cos_sim:.4f} Anisotropy={anisotropy:.4f} DaviesBouldin={db_score:.4f} KNNAcc={knn_acc:.2f}±{knn_acc_std:.2f}")

    results[model] = pd.DataFrame(rows)


In [ ]:
for model in models:
    results[model].to_csv(f"{model}_TPCF_id_scores.csv", index=False)

pd.concat(results, names=["model"])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import string

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

COLORS = {"byol": "#1f4e79", "dino": "#b5442b"}

metrics = [
    ("tpcf_score", "TPCF score", "tpcf_std"),
    ("id_mean", "Intrinsic dimension", "id_std"),
    ("avg_cosine_similarity", "Avg. cosine similarity", "cosine_similarity_std"),
    ("anisotropy", "Anisotropy (top-1 EVR)", None),
    ("davies_bouldin", "Davies-Bouldin score", None),
    ("knn_accuracy", "KNN accuracy (%)", "knn_accuracy_std"),
]


def plot_gradient_band(ax, x, y, y_err, color, n_layers=40, max_sigma=3.0):
    """Shade the uncertainty band with an alpha gradient that fades away from the mean."""
    x = np.asarray(x)
    y = np.asarray(y)
    y_err = np.asarray(y_err)
    sigmas = np.linspace(max_sigma / n_layers, max_sigma, n_layers)
    for sigma in sigmas[::-1]:
        alpha = 0.35 * np.exp(-0.5 * sigma ** 2)
        ax.fill_between(x, y - sigma * y_err, y + sigma * y_err, color=color, alpha=alpha, linewidth=0)


n_cols = 3
n_rows = -(-len(metrics) // n_cols)  # ceil division

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3.3 * n_cols, 2.5 * n_rows), squeeze=False, constrained_layout=True
)
axes = axes.flatten()

for panel_label, ax, (col, title, err_col) in zip(string.ascii_lowercase, axes, metrics):
    for model in models:
        res, color = results[model], COLORS[model]
        ax.plot(res["epoch"], res[col], color=color, label=model.upper())
        if col == "tpcf_score":
            plot_gradient_band(ax, res["epoch"], res[col], res[err_col], color=color)
        elif err_col is not None:
            ax.fill_between(
                res["epoch"],
                res[col] - res[err_col],
                res[col] + res[err_col],
                color=color,
                alpha=0.2,
            )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.text(-0.15, 1.08, f"({panel_label})", transform=ax.transAxes, fontsize=11, fontweight="bold")

axes[0].legend()

# hide any unused axes in the grid
for ax in axes[len(metrics):]:
    ax.axis("off")

fig.savefig("Self_model_metrics_grid.pdf", bbox_inches="tight")
fig.savefig("Self_model_metrics_grid.png", bbox_inches="tight")
plt.show()